In [1]:
import numpy as np
import cv2 as cv
import matplotlib.pyplot as plt
import pandas as pd
import os

import cvTools.interaction as cvi
import cvTools.segmentation as cvs
import extractor as ex
from tqdm import tqdm

In [34]:
data_path = os.path.expanduser('~/uni/Computer-Vision/data/')
data = pd.read_csv(f'{data_path}ISIC_2019_GroundTruth.csv', index_col='image', dtype={'image':str, 'MEL':bool, 'NV':bool}).drop(columns='NV')
for Type in ('MEL/', 'NV/'):
    with os.scandir(data_path + Type) as Dir:
        for entry in Dir:
            print(entry.path)

/home/teo/uni/Computer-Vision/data/MEL/ISIC_0000002.jpg
/home/teo/uni/Computer-Vision/data/MEL/ISIC_0000004.jpg
/home/teo/uni/Computer-Vision/data/MEL/ISIC_0000013.jpg
/home/teo/uni/Computer-Vision/data/MEL/ISIC_0000022_downsampled.jpg
/home/teo/uni/Computer-Vision/data/MEL/ISIC_0000026_downsampled.jpg
/home/teo/uni/Computer-Vision/data/MEL/ISIC_0000029_downsampled.jpg
/home/teo/uni/Computer-Vision/data/MEL/ISIC_0000030_downsampled.jpg
/home/teo/uni/Computer-Vision/data/MEL/ISIC_0000031_downsampled.jpg
/home/teo/uni/Computer-Vision/data/MEL/ISIC_0000035_downsampled.jpg
/home/teo/uni/Computer-Vision/data/MEL/ISIC_0000036_downsampled.jpg
/home/teo/uni/Computer-Vision/data/MEL/ISIC_0000040_downsampled.jpg
/home/teo/uni/Computer-Vision/data/MEL/ISIC_0000043_downsampled.jpg
/home/teo/uni/Computer-Vision/data/MEL/ISIC_0000046_downsampled.jpg
/home/teo/uni/Computer-Vision/data/MEL/ISIC_0000049_downsampled.jpg
/home/teo/uni/Computer-Vision/data/MEL/ISIC_0000054_downsampled.jpg
/home/teo/uni/Co

In [64]:
L_m = []
L_s = []
A_m = []
A_s = []
B_m = []
B_s = []
height = []
width = []
ver_symm = []
hor_symm = []
cent_symm = []
feat_num = []


img_path = 'data/MEL/ISIC_0000043_downsampled.jpg'
# def main(img_path):
img = cv.imread(img_path)[1:-1, 1:-1, :]  # some photos have white borders
lab = cv.cvtColor(img, cv.COLOR_BGR2LAB)

### trimming hair
lab = cv.dilate(lab, np.ones((3,3), dtype=np.uint8), iterations=2)
lab = cv.erode(lab, np.ones((3,3), dtype=np.uint8), iterations=2)

### CLAHE
clahe = cv.createCLAHE(clipLimit=1, tileGridSize=(10,10))
lab[..., 0] = clahe.apply(lab[..., 0])
bgr = cv.cvtColor(lab, cv.COLOR_LAB2BGR)
# cvi.show([img, cl], ['img', 'cl'])


## h thresholding
otsu, bgr = cvs.thresholding(bgr)

### kill small patches
conts, _ = cv.findContours(otsu, cv.RETR_LIST, cv.CHAIN_APPROX_SIMPLE)
contSave = cvs.counter(otsu, 1000)
if len(contSave) == 0: contSave = cvs.counter(otsu, 100)

otsu = np.zeros_like(otsu)
for cont in contSave: cv.drawContours(otsu, conts, cont, 255, -1)
rows = [cv.hasNonZero(otsu[i]) for i in range(otsu.shape[0])]
cols = [cv.hasNonZero(otsu[:,i]) for i in range(otsu.shape[1])]
bgr = (bgr[rows][:, cols]).copy()
otsu = (otsu[rows][:, cols]).copy()
bgr = cv.bitwise_and(bgr, bgr, mask=otsu)
lab = cv.cvtColor(bgr, cv.COLOR_BGR2LAB)

### Filtering
lab = cv.bilateralFilter(lab, d=30, sigmaColor=20, sigmaSpace=10)
sift = cv.SIFT_create()
kp, des = sift.detectAndCompute(lab[..., 0], otsu)
# bgr = cv.cvtColor(lab, cv.COLOR_LAB2BGR)
# out = np.copy(bgr)
# out = cv.drawKeypoints(bgr, kp, out, flags=cv.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
# cvi.show([img, out], ['img', 'kp'])
means, stds = cv.meanStdDev(lab, mask=otsu)
active = cv.countNonZero(otsu)
vert = cv.flip(otsu, 0)
hor = cv.flip(otsu, 1)
center = cv.flip(hor, 0)
for axis, res in zip((vert, hor, center), (ver_symm, hor_symm, cent_symm)): res.append(cv.countNonZero(cv.bitwise_and(otsu, axis)) / active)
for i, channel in enumerate((L_m, A_m, B_m)): channel.append(means[i].item())
for i, channel in enumerate((L_s, A_s, B_s)): channel.append(stds[i].item())
height.append(otsu.shape[0])
width.append(otsu.shape[1])
feat_num.append(len(kp))

In [65]:
cvi.show([otsu])

In [44]:
means

array([[ 96.86333198],
       [153.28978841],
       [147.34134914]])

In [75]:
pd.DataFrame({
    'L_m': means[0],
    'L_s': stds[0],
    'A_m': means[1],
    'A_s': stds[1],
    'B_m': means[2],
    'B_s': stds[2]
    }, index=['ISIC_0012549_downsampled'])

,L_m,L_s,A_m,A_s,B_m,B_s
ISIC_0012549_downsampled,104.55938,34.166983,143.683893,6.005756,143.82989,8.142614


In [43]:
data

,MEL,boh
image,,
ISIC_0000000,False,NaN
ISIC_0000001,False,NaN
ISIC_0000002,True,NaN
ISIC_0000003,False,NaN
ISIC_0000004,True,NaN
...,...,...
ISIC_0073241,True,NaN
ISIC_0073244,False,NaN
ISIC_0073245,False,NaN


In [42]:
data.loc['ISIC_0012549_downsampled', 'boh'] = 2

In [36]:
cvi.show([img, cv.cvtColor(lab, cv.COLOR_LAB2BGR)], ['original', 'final'])

In [5]:
%%timeit
img_path = 'data/MEL/ISIC_0000140_downsampled.jpg'
img = cv.imread(img_path)[1:-1, 1:-1, :]  # some photos have white borders

lab = cv.cvtColor(img, cv.COLOR_BGR2LAB)
### trimming hair
lab = cv.dilate(lab, np.ones((3,3), dtype=np.uint8), iterations=2)
lab = cv.erode(lab, np.ones((3,3), dtype=np.uint8), iterations=2)

### CLAHE
clahe = cv.createCLAHE(clipLimit=1, tileGridSize=(10,10))
lab[..., 0] = clahe.apply(lab[..., 0])
bgr = cv.cvtColor(lab, cv.COLOR_LAB2BGR)
# cvi.show([img, cl], ['img', 'cl'])


## h thresholding
temp = cv.cvtColor(bgr, cv.COLOR_BGR2HSV)  # to img?
_, thresh = cv.threshold(temp[..., 0], 23, 255, cv.THRESH_BINARY_INV)
_, thresh2 = cv.threshold(temp[..., 0], 110, 255, cv.THRESH_BINARY)
thresh = cv.bitwise_or(thresh, thresh2)

### Segmentation
temp = cv.cvtColor(bgr, cv.COLOR_BGR2GRAY)
lens = np.zeros_like(temp)
cv.thresholdWithMask(temp, lens, thresh, 10, 255, cv.THRESH_BINARY)  # lens is the new mask of active pxs

rows = [cv.hasNonZero(lens[i]) for i in range(lens.shape[0])]
cols = [cv.hasNonZero(lens[:, i]) for i in range(lens.shape[1])]
lens = (lens[rows][:, cols])[40:-40, 40:-40].copy()
temp = (temp[rows][:, cols])[40:-40, 40:-40].copy()
bgr = (bgr[rows][:, cols])[40:-40, 40:-40, :]

## otsu
otsu = np.zeros_like(temp)
cv.thresholdWithMask(temp, otsu, lens, 0, 255, cv.THRESH_BINARY_INV + cv.THRESH_OTSU)

### kill small patches
conts, _ = cv.findContours(otsu, cv.RETR_LIST, cv.CHAIN_APPROX_SIMPLE)
cxs = []
cys = []
contSave = []
for i, cont in enumerate(conts):
    mom = cv.moments(cont)
    if mom['m00'] < 1000: continue  # Area    # not guf  or mom['m00'] > 15e4  might wanna go little by little down
    cx = int(mom['m10'] / mom['m00'])  # centroid position
    lim = otsu.shape[1] * .1
    if cx < lim or cx > otsu.shape[1] - lim: continue
    cy = int(mom['m01'] / mom['m00'])
    lim = otsu.shape[0] * .1
    if cy < lim or cy > otsu.shape[0] - lim: continue
    cxs.append(cx)
    cys.append(cy)
    contSave.append(i)

if len(contSave) != 0:
    otsu = np.zeros_like(otsu)
    for cont in contSave: cv.drawContours(otsu, conts, cont, 255, -1)
    rows = [cv.hasNonZero(otsu[i]) for i in range(otsu.shape[0])]
    cols = [cv.hasNonZero(otsu[:,i]) for i in range(otsu.shape[1])]
    bgr = (bgr[rows][:, cols]).copy()
    otsu = (otsu[rows][:, cols]).copy()
    seg = cv.bitwise_and(bgr, bgr, mask=otsu)
    segL = cv.cvtColor(seg, cv.COLOR_BGR2LAB)
    
    ### Filtering
    lab = cv.bilateralFilter(segL, d=30, sigmaColor=20, sigmaSpace=10)
else: print('Damn')
orb = cv.SIFT_create()
bgr = cv.cvtColor(lab, cv.COLOR_LAB2BGR)
kp, des = orb.detectAndCompute(lab[..., 0], otsu)
# cvi.show([img, cl_BGR], ['original', 'pre-processed'])
out = np.copy(bgr)
out = cv.drawKeypoints(bgr, kp, out, flags=cv.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
# cvi.show([img, out], ['img', 'kp'])
means, stds = cv.meanStdDev(lab, mask=otsu)

322 ms ± 17.5 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [35]:
prova = otsu
active = cv.countNonZero(prova)
vert = cv.flip(prova, 0)
hor = cv.flip(prova, 1)
center = cv.flip(hor, 0)
for i in (vert, hor, center): print(cv.countNonZero(cv.bitwise_and(prova, i)) / active)

0.8298865910607072
0.6630522912711038
0.6532868065890081


In [37]:
otsu.shape

(594, 563)

# Pandas

In [8]:
import pandas as pd
import os

In [18]:
data = pd.read_csv('data/ISIC_2019_GroundTruth.csv', index_col='image', dtype={'image':str, 'MEL':bool, 'NV':bool}).drop(columns='NV')
# with os.scandir('data/MEL/') as Dir:
    # for entry in Dir:
# data[image]

In [26]:
data.index

Index(['ISIC_0000000', 'ISIC_0000001', 'ISIC_0000002', 'ISIC_0000003',
       'ISIC_0000004', 'ISIC_0000006', 'ISIC_0000007', 'ISIC_0000008',
       'ISIC_0000009', 'ISIC_0000010',
       ...
       'ISIC_0073231', 'ISIC_0073232', 'ISIC_0073237', 'ISIC_0073238',
       'ISIC_0073240', 'ISIC_0073241', 'ISIC_0073244', 'ISIC_0073245',
       'ISIC_0073249', 'ISIC_0073251'],
      dtype='object', name='image', length=17397)

In [25]:
type(data.loc['ISIC_0012549_downsampled'])

pandas.core.series.Series

In [82]:
img_path = 'data/MEL/ISIC_0000022_downsampled.jpg'
# def main(img_path):
img = cv.imread(img_path)[1:-1, 1:-1, :]  # some photos have white borders
lab = cv.cvtColor(img, cv.COLOR_BGR2LAB)

### trimming hair
lab = cv.dilate(lab, np.ones((3,3), dtype=np.uint8), iterations=2)
lab = cv.erode(lab, np.ones((3,3), dtype=np.uint8), iterations=2)

### CLAHE
clahe = cv.createCLAHE(clipLimit=1, tileGridSize=(10,10))
lab[..., 0] = clahe.apply(lab[..., 0])
bgr = cv.cvtColor(lab, cv.COLOR_LAB2BGR)
# cvi.show([img, cl], ['img', 'cl'])


## h thresholding
otsu, bgr = cvs.thresholding(bgr)

### kill small patches
conts, _ = cv.findContours(otsu, cv.RETR_LIST, cv.CHAIN_APPROX_SIMPLE)
contSave = cvs.counter(otsu, 1000)
if len(contSave) == 0: contSave = cvs.counter(otsu, 100)

otsu = np.zeros_like(otsu)
for cont in contSave: cv.drawContours(otsu, conts, cont, 255, -1)
rows = [cv.hasNonZero(otsu[i]) for i in range(otsu.shape[0])]
cols = [cv.hasNonZero(otsu[:,i]) for i in range(otsu.shape[1])]
bgr = (bgr[rows][:, cols]).copy()
otsu = (otsu[rows][:, cols]).copy()
bgr = cv.bitwise_and(bgr, bgr, mask=otsu)
lab = cv.cvtColor(bgr, cv.COLOR_BGR2LAB)

### Filtering
lab = cv.bilateralFilter(lab, d=30, sigmaColor=20, sigmaSpace=10)
sift = cv.SIFT_create()
# kp, des = sift.detectAndCompute(lab[..., 0], otsu)
kp = sift.detect(lab[..., 0], otsu)
bgr = cv.cvtColor(lab, cv.COLOR_LAB2BGR)
out = np.copy(bgr)
out = cv.drawKeypoints(bgr, kp, out, flags=cv.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
cvi.show([img, out], ['img', 'kp'])
means, stds = cv.meanStdDev(lab, mask=otsu)